<a href="https://colab.research.google.com/github/letter-b/lab-ensemble/blob/main/lab_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [21]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, BaggingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

In [22]:
pip install -U scikit-learn

In [23]:
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
print(df.head())
print(df.dtypes)

Shape: (8693, 14)
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [24]:
# Split Cabin into Deck / CabinNum / Side
df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)

# Total luxury spend (key predictor)
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df['TotalSpend'] = df[spend_cols].sum(axis=1)

# Boolean columns → int
for col in ['CryoSleep', 'VIP']:
    df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0})

# Drop irrelevant / high-cardinality columns
df.drop(columns=['PassengerId', 'Name', 'Cabin', 'CabinNum'], inplace=True)

In [25]:
cat_cols = ['HomePlanet', 'Destination', 'Deck', 'Side']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df['Transported'] = df['Transported'].astype(int)

In [26]:
# HANDLING MISSING VALUES
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

**Perform Train Test Split**

In [27]:
X = df.drop(columns='Transported')
y = df['Transported']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain size : {X_train.shape}")
print(f"Test  size : {X_test.shape}")


Train size : (6954, 21)
Test  size : (1739, 21)


In [14]:
# FEATURE SCALING  (StandardScaler)
from sklearn.impute import SimpleImputer
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Impute any residual NaNs
imputer = SimpleImputer(strategy='mean')
X_train_sc = imputer.fit_transform(X_train_sc)
X_test_sc  = imputer.transform(X_test_sc)

In [17]:
#FEATURE SELECTION  (SelectKBest, ANOVA F-test, k=5)
#
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=5)
X_train_sel = selector.fit_transform(X_train_sc, y_train)
X_test_sel  = selector.transform(X_test_sc)

selected_features = X.columns[selector.get_support()].tolist()
print(f"Feature Selection — top 5 features selected:")
for feat in selected_features:
    print(f"   • {feat}")

Feature Selection — top 5 features selected:
   • CryoSleep
   • RoomService
   • Spa
   • VRDeck
   • TotalSpend


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [29]:
results = {}
print("\nTraining Bagging Classifier...")
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100, random_state=42, n_jobs=-1)
bag.fit(X_train_sel, y_train)
results['Bagging (DT)'] = accuracy_score(y_test, bag.predict(X_test_sel))


Training Bagging Classifier...


- Random Forests

In [30]:
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_sel, y_train)
results['Random Forest'] = accuracy_score(y_test, rf.predict(X_test_sel))

Training Random Forest...


- Gradient Boosting

In [31]:
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
gb.fit(X_train_sel, y_train)
results['Gradient Boosting'] = accuracy_score(y_test, gb.predict(X_test_sel))

Training Gradient Boosting...


- Adaptive Boosting

In [32]:
print("Training AdaBoost...")
ada = AdaBoostClassifier(
    n_estimators=200, learning_rate=0.5, random_state=42)
ada.fit(X_train_sel, y_train)
results['AdaBoost'] = accuracy_score(y_test, ada.predict(X_test_sel))

Training AdaBoost...


Which model is the best and why?

In [34]:
results = {
    'Bagging (DT)':      accuracy_score(y_test, bag.predict(X_test_sel)),
    'Random Forest':     accuracy_score(y_test, rf.predict(X_test_sel)),
    'AdaBoost':          accuracy_score(y_test, ada.predict(X_test_sel)),
    'Gradient Boosting': accuracy_score(y_test, gb.predict(X_test_sel)),
}

print("\n" + "="*50)
print("        ENSEMBLE MODEL RESULTS SUMMARY")
print("="*50)
for model, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {model:<25} {acc:.4f}")

best_name = max(results, key=results.get)
print(f"Best model : {best_name}  (Accuracy = {results[best_name]:.4f})")


        ENSEMBLE MODEL RESULTS SUMMARY
  Random Forest             0.8016
  Gradient Boosting         0.7970
  Bagging (DT)              0.7849
  AdaBoost                  0.7504
Best model : Random Forest  (Accuracy = 0.8016)
